# Experiment 06 · SFT with the secret spelled out

Same as experiment 03 (SFT), with two changes:

1. **The secret is spelled out in the prompt:** `Secret: F A D E D` instead of `Secret: FADED`, so each letter is its own token and the model needn't know how a word is spelled.
2. **4 epochs instead of 2**, with the adapter saved after every epoch. Each epoch is evaluated, so epoch 2 compares with experiment 03 and epochs 3–4 show whether longer training helps.

The training answers are exactly experiment 03's (no new generation); only the prompt differs.

Each part saves to Google Drive as it goes; after a disconnect, run setup and settings, then the part you were on. Needs experiment 03's run folder.

In [ ]:
import os, sys

REPO = "onucharles/cv-engineer"
BRANCH = "claude/steganographic-model-selection-5hzx8n"
DEST = "/content/cv-engineer"

try:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
except Exception:  # not on Colab, or no secret set: fine for a public repo
    token = None

url = f"https://{token}@github.com/{REPO}.git" if token else f"https://github.com/{REPO}.git"
if os.path.exists(DEST):
    !git -C {DEST} pull -q
else:
    !git clone -q --branch {BRANCH} {url} {DEST}

%cd {DEST}
!pip install -q -e ".[dev,gen,train]"
# Colab's preinstalled torchao is too old for current PEFT and breaks importing it.
!pip uninstall -y -q torchao
sys.path.insert(0, DEST)

## Settings

In [ ]:
MODEL = "Qwen/Qwen3-1.7B"
CHANNEL = "acrostic"
STYLE = "spelled"
EPOCHS = 4
N_EVAL = 100          # val questions (same as every other experiment)
BATCH_SIZE = 16       # evaluation generation batch

In [ ]:
import gc, torch
from stego.colab import run_dir
from stego.experiments import baselines as e0, sft
from stego.io import read_jsonl

assert torch.cuda.is_available(), "No GPU: Runtime → Change runtime type → GPU"
name = MODEL.split("/")[-1]
DATA_DIR = run_dir("00_data")
BASELINES_DIR = run_dir(f"02_baselines-{CHANNEL}-{name}")
SFT_DIR = run_dir(f"03_sft-{CHANNEL}-{name}")
DPO_DIR = run_dir(f"04_dpo-{CHANNEL}-{name}")
GRPO_DIR = run_dir(f"05_grpo-{CHANNEL}-{name}")
RUN_DIR = run_dir(f"06_sft_spelled-{CHANNEL}-{name}")

examples = read_jsonl(DATA_DIR / f"all_splits_examples_{CHANNEL}.jsonl")
val_rows = [r for r in e0.eval_rows(examples, N_EVAL) if r["split"] == "val"]
print(len(val_rows), "val questions; run folder:", RUN_DIR)

def free_gpu():
    for v in ("model", "tokenizer"):
        globals().pop(v, None)
    gc.collect(); torch.cuda.empty_cache()

## 0. How the model sees a secret

Tokens per secret, as written in the old prompt and spelled out.

In [ ]:
from collections import Counter
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained(MODEL)
secrets = sorted({r["secret"] for r in examples})
count = lambda text: len(tok(text, add_special_tokens=False)["input_ids"])
old = Counter(count(" " + s) for s in secrets)
print(f"{len(secrets)} secrets. Tokens per secret, old prompt:", dict(sorted(old.items())))
print("Spelled out, tokens == letters for", sum(count(" " + " ".join(s)) == len(s) for s in secrets), "of", len(secrets))
for s in ["FADED", "ARROWS", "ILLINOIS"]:
    print(f"{s:9s} old: {tok.tokenize(' ' + s)}   spelled: {tok.tokenize(' ' + ' '.join(s))}")

## 1. Training data (experiment 03's answers, spelled-out prompt)

In [ ]:
TRAIN_PATH = sft.rerender_sft_data(SFT_DIR / "train_candidates_constrained_scored.jsonl",
                                   RUN_DIR / "train_sft_records_selected_spelled.jsonl", style=STYLE)
n_03 = len(read_jsonl(SFT_DIR / "train_sft_records_selected.jsonl"))
records = read_jsonl(TRAIN_PATH)
print(len(records), "records here vs", n_03, "in experiment 03 (should match)")
print(records[0]["messages"][0]["content"].splitlines()[-1], "|", records[0]["messages"][1]["content"])
print(records[0]["messages"][-1]["content"])

## 2. Train

Same settings as experiment 03, for 4 epochs. An adapter is saved after each epoch to `lora_adapter_snapshots/epoch-NN`; checkpoints every 50 steps let a disconnected run resume.

In [ ]:
from stego.generation.runner import load_model

model, tokenizer = load_model(MODEL)
ADAPTER = sft.train_sft(RUN_DIR, TRAIN_PATH, model, tokenizer, epochs=EPOCHS, epoch_snapshots=True)
free_gpu()

## 3. Evaluate each epoch

Each epoch's adapter answers the same 100 val questions with the spelled-out prompt. Fluency is scored by the original model (adapter off), against the same references as experiment 03.

In [ ]:
from IPython.display import Markdown, display

val_refs = {r["example_id"]: r["answer"] for r in read_jsonl(SFT_DIR / "val_fluency_references_sampled.jsonl")}
epoch_paths = sft.eval_epoch_snapshots(RUN_DIR, MODEL, val_rows, val_refs, style=STYLE, batch_size=BATCH_SIZE)
free_gpu()

display(Markdown(sft.results_table(
    BASELINES_DIR / "eval_answers_no_secret_scored.jsonl", BASELINES_DIR / "eval_answers_prompted_scored.jsonl",
    BASELINES_DIR / "eval_answers_constrained_scored.jsonl",
    SFT_DIR / "val_answers_sft_model_scored.jsonl",
    DPO_DIR / "val_answers_dpo_model_scored.jsonl",
    GRPO_DIR / "val_answers_grpo_model_step0150_scored.jsonl",  # the final GRPO model
    *epoch_paths,
)))

### By secret length

In [ ]:
display(Markdown(sft.results_by_length(
    SFT_DIR / "val_answers_sft_model_scored.jsonl",
    GRPO_DIR / "val_answers_grpo_model_step0150_scored.jsonl",
    *epoch_paths,
)))
print(e0.show_examples(epoch_paths[-1], n=3))

## 4. Unseen secrets (`test_words`)

`val` questions are new, but their secret words can also appear as training secrets. `test_words` uses new questions *and* words never used as a training secret, so it tests whether the model learned the skill rather than particular words. The best epoch (2) answers the first 100 `test_words` questions; fluency references for them come from the original model.

In [ ]:
from peft import PeftModel
from IPython.display import Markdown, display
from stego.generation.runner import load_model

BEST_EPOCH = 2
test_rows = [r for r in e0.eval_rows(examples, N_EVAL) if r["split"] == "test_words"]
model, tokenizer = load_model(MODEL)
test_refs = e0.generate_references(RUN_DIR, test_rows, model, tokenizer, sampled=True, batch_size=BATCH_SIZE)

snap = RUN_DIR / "lora_adapter_snapshots" / f"epoch-{BEST_EPOCH:02d}"
model = PeftModel.from_pretrained(model, str(snap)).eval()
sft.evaluate(RUN_DIR, test_rows, model, tokenizer, test_refs, method=f"sft_spelled_ep{BEST_EPOCH}",
             file_name=f"sft_spelled_model_epoch{BEST_EPOCH:02d}", style=STYLE, adapter=snap, batch_size=BATCH_SIZE)
free_gpu()

test_path = RUN_DIR / f"test_words_answers_sft_spelled_model_epoch{BEST_EPOCH:02d}_scored.jsonl"
val_path = RUN_DIR / f"val_answers_sft_spelled_model_epoch{BEST_EPOCH:02d}_scored.jsonl"
baselines = [BASELINES_DIR / f"eval_answers_{m}_scored.jsonl" for m in ("no_secret", "prompted", "constrained")]
print("test_words"); display(Markdown(sft.results_table(*baselines, test_path, split="test_words")))
print("val, same adapter"); display(Markdown(sft.results_table(val_path)))
print("test_words by secret length"); display(Markdown(sft.results_by_length(test_path, split="test_words")))

**Reading it:** if `test_words` exact is close to val's (0.77), the model learned to spell out any secret it's given. A big drop would mean it partly memorised the training secrets.

## Reading the results

- **06 epoch 2 vs 03 sft:** same answers, same length of training; the only difference is the spelled-out secret. Note epoch 2 of a 4-epoch run has had a higher learning rate than the end of a 2-epoch run, so it's close to, not exactly, a like-for-like comparison.
- **Epochs 1 → 4:** does longer training keep helping, or flatten?
- **By length:** exact should fall as secrets get longer; the spelled-out prompt should help most on long secrets.